# Analyzing the Trade-off Between Style Fidelity and Content Preservation in AdaIN-Based Neural Style Transfer under Different Style Strengths

---

## Abstract

Adaptive Instance Normalization (AdaIN; Huang & Belongie, 2017) performs arbitrary style transfer in a single feed-forward pass by aligning the channel-wise mean and standard deviation of content features with those of a style image. A key property of AdaIN is the **content–style trade-off parameter** $\alpha \in [0, 1]$, which interpolates in feature space between the original content features and the stylized (AdaIN) features:

$$
t = \text{AdaIN}\big(f(c), f(s)\big), \qquad
T(c, s, \alpha) = g\big(\alpha \cdot t + (1 - \alpha) \cdot f(c)\big)
$$

where $f(\cdot)$ is a fixed VGG-19 encoder (up to `relu4_1`) and $g(\cdot)$ is a learned decoder.

This notebook studies how **style fidelity** and **content preservation** change as $\alpha$ varies, using several content–style pairs and complementary quantitative metrics.

## Research Questions

| ID | Question |
| :-- | :-- |
| **RQ1** | How do style fidelity and content preservation change as $\alpha$ increases from 0 to 1? |
| **RQ2** | Is the trade-off linear, or is there a "knee" where content degrades faster than style improves? |
| **RQ3** | Does the optimal (balanced) $\alpha$ depend on the style image? |

## Hypotheses

- **H1:** Style loss decreases monotonically as $\alpha$ increases.
- **H2:** Content loss increases monotonically, and structural similarity (SSIM) decreases, as $\alpha$ increases.
- **H3:** The trade-off is non-linear, so a balanced $\alpha^*$ exists that sits strictly between 0 and 1 and varies across styles.

## How to Use This Notebook

1. Set **`ALPHA`** in the **Configuration** cell (Section 1) to try one style strength.
2. Use the **interactive slider** (Section 6) to explore $\alpha$ in real time.
3. Run the **α-sweep experiment** (Section 8) to produce the quantitative trade-off analysis. Results are saved to `results/adain/`.

## 1. Configuration

> **Change `ALPHA` here.** `0.0` = no stylization (content reconstruction only), `1.0` = full AdaIN stylization.

In [1]:
# =========================
#   EXPERIMENT CONFIGURATION
# =========================

# --- Single-run style strength (edit this!) ---
ALPHA = 0.6                       # float in [0.0, 1.0]

# --- Alpha values used in the systematic sweep (Section 8) ---
ALPHA_SWEEP = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# --- Images (filenames inside data/content and data/style) ---
SINGLE_CONTENT = "bird.jpg"
SINGLE_STYLE = "picasso.jpg"

SWEEP_CONTENTS = ["bird.jpg", "dancing.jpg", "amazon.jpg"]
SWEEP_STYLES = ["picasso.jpg", "second.jpg", "third.webp"]

# --- Image processing ---
IMAGE_SIZE = 512                  # longest side in pixels (lower this if you run out of memory)

# --- Reproducibility & output ---
SEED = 42
SAVE_RESULTS = True               # save images, CSV and figures to results/adain/

assert 0.0 <= ALPHA <= 1.0, "ALPHA must be in [0, 1]"
assert all(0.0 <= a <= 1.0 for a in ALPHA_SWEEP), "All ALPHA_SWEEP values must be in [0, 1]"

## 2. Environment Setup

In [2]:
import os
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

PyTorch 2.6.0+cu124 | device: cuda


In [3]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    """Walk upward until a folder containing data/content and data/style is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "content").is_dir() and (candidate / "data" / "style").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root (folder containing data/content and data/style).")


ROOT = find_project_root()
CONTENT_DIR = ROOT / "data" / "content"
STYLE_DIR = ROOT / "data" / "style"
MODEL_DIR = ROOT / "models" / "adain"
RESULT_DIR = ROOT / "results" / "adain"
FIG_DIR = RESULT_DIR / "figures"
IMG_OUT_DIR = RESULT_DIR / "images"

for d in (MODEL_DIR, FIG_DIR, IMG_OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Content imgs :", sorted(p.name for p in CONTENT_DIR.iterdir()))
print("Style imgs   :", sorted(p.name for p in STYLE_DIR.iterdir()))

Project root : d:\MAIN DATA\Documents\Neural Style Transfer
Content imgs : ['a.jpg', 'a.png', 'amazon.jpg', 'bird.jpg', 'dancing.jpg', 'someLogo.jpg']
Style imgs   : ['first.webp', 'picasso.jpg', 'second.jpg', 'ss.png', 'third.webp']


## 3. Pretrained AdaIN Weights

AdaIN needs a decoder trained on MS-COCO (content) + WikiArt (style). Training one takes many GPU hours, so we use the publicly released weights from the widely used PyTorch port [naoto0804/pytorch-AdaIN](https://github.com/naoto0804/pytorch-AdaIN), which reproduces the original Torch implementation by Huang & Belongie.

| File | Role |
| :-- | :-- |
| `vgg_normalised.pth` | VGG-19 encoder with built-in input normalization |
| `decoder.pth` | Decoder trained to invert AdaIN features |

The weights are downloaded once into `models/adain/`. This folder is git-ignored via the `*.pth` rule.

In [4]:
WEIGHT_URLS = {
    "vgg_normalised.pth": "https://github.com/naoto0804/pytorch-AdaIN/releases/download/v0.0.0/vgg_normalised.pth",
    "decoder.pth": "https://github.com/naoto0804/pytorch-AdaIN/releases/download/v0.0.0/decoder.pth",
}

for fname, url in WEIGHT_URLS.items():
    dest = MODEL_DIR / fname
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[cached]      {fname} ({dest.stat().st_size / 1e6:.1f} MB)")
        continue
    print(f"[downloading] {fname} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"              done ({dest.stat().st_size / 1e6:.1f} MB)")

[cached]      vgg_normalised.pth (80.1 MB)
[cached]      decoder.pth (14.0 MB)


## 4. Model Architecture

The architecture follows the original paper exactly:

- **Encoder** $f$: VGG-19 truncated at `relu4_1` (frozen).
- **AdaIN layer**: $\text{AdaIN}(x, y) = \sigma(y)\left(\dfrac{x - \mu(x)}{\sigma(x)}\right) + \mu(y)$
- **Decoder** $g$: mirrors the encoder. Pooling is replaced by nearest-neighbor upsampling, and reflection padding is used to avoid border artifacts.

In [5]:
def build_vgg() -> nn.Sequential:
    """Full normalised VGG-19 (53 layers) matching vgg_normalised.pth."""
    def conv(cin, cout):
        return [nn.ReflectionPad2d((1, 1, 1, 1)), nn.Conv2d(cin, cout, (3, 3)), nn.ReLU()]

    def pool():
        return [nn.MaxPool2d((2, 2), (2, 2), (0, 0), ceil_mode=True)]

    layers = [nn.Conv2d(3, 3, (1, 1))]
    layers += conv(3, 64) + conv(64, 64) + pool()                                         # block 1
    layers += conv(64, 128) + conv(128, 128) + pool()                                     # block 2
    layers += conv(128, 256) + conv(256, 256) + conv(256, 256) + conv(256, 256) + pool()  # block 3
    layers += conv(256, 512) + conv(512, 512) + conv(512, 512) + conv(512, 512) + pool()  # block 4
    layers += conv(512, 512) + conv(512, 512) + conv(512, 512) + conv(512, 512)           # block 5
    return nn.Sequential(*layers)


def build_decoder() -> nn.Sequential:
    """Decoder that inverts relu4_1 features back to RGB (matches decoder.pth)."""
    def conv(cin, cout, relu=True):
        block = [nn.ReflectionPad2d((1, 1, 1, 1)), nn.Conv2d(cin, cout, (3, 3))]
        return block + ([nn.ReLU()] if relu else [])

    up = lambda: [nn.Upsample(scale_factor=2, mode="nearest")]
    layers = conv(512, 256) + up()
    layers += conv(256, 256) + conv(256, 256) + conv(256, 256) + conv(256, 128) + up()
    layers += conv(128, 128) + conv(128, 64) + up()
    layers += conv(64, 64) + conv(64, 3, relu=False)
    return nn.Sequential(*layers)


vgg_full = build_vgg()
vgg_full.load_state_dict(torch.load(MODEL_DIR / "vgg_normalised.pth", map_location="cpu"))
decoder = build_decoder()
decoder.load_state_dict(torch.load(MODEL_DIR / "decoder.pth", map_location="cpu"))

# Encoder slices used both for stylization (relu4_1) and for loss metrics (relu1_1 ... relu4_1)
vgg_layers = list(vgg_full.children())[:31]
ENC_SLICES = nn.ModuleList([
    nn.Sequential(*vgg_layers[:4]),     # -> relu1_1
    nn.Sequential(*vgg_layers[4:11]),   # -> relu2_1
    nn.Sequential(*vgg_layers[11:18]),  # -> relu3_1
    nn.Sequential(*vgg_layers[18:31]),  # -> relu4_1
]).to(DEVICE).eval()
decoder = decoder.to(DEVICE).eval()

for p in list(ENC_SLICES.parameters()) + list(decoder.parameters()):
    p.requires_grad_(False)

print("Encoder & decoder loaded.")
print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()) / 1e6:.2f} M")

Encoder & decoder loaded.
Decoder parameters: 3.51 M


## 5. AdaIN Operation & Style-Strength Control

In [6]:
def calc_mean_std(feat: torch.Tensor, eps: float = 1e-5):
    """Channel-wise mean and std over spatial dimensions. Shapes: (N, C, 1, 1)."""
    n, c = feat.shape[:2]
    flat = feat.reshape(n, c, -1)
    mean = flat.mean(dim=2).reshape(n, c, 1, 1)
    std = (flat.var(dim=2) + eps).sqrt().reshape(n, c, 1, 1)
    return mean, std


def adain(content_feat: torch.Tensor, style_feat: torch.Tensor) -> torch.Tensor:
    c_mean, c_std = calc_mean_std(content_feat)
    s_mean, s_std = calc_mean_std(style_feat)
    return (content_feat - c_mean) / c_std * s_std + s_mean


@torch.no_grad()
def encode(img: torch.Tensor, all_layers: bool = False):
    """Return relu4_1 features, or a list [relu1_1, relu2_1, relu3_1, relu4_1] if all_layers=True."""
    feats, x = [], img
    for block in ENC_SLICES:
        x = block(x)
        feats.append(x)
    return feats if all_layers else feats[-1]


@torch.no_grad()
def stylize(content: torch.Tensor, style: torch.Tensor, alpha: float = 1.0,
            content_feat: torch.Tensor = None, style_feat: torch.Tensor = None) -> torch.Tensor:
    """
    AdaIN style transfer with style strength alpha:
        T(c, s, alpha) = g( alpha * AdaIN(f(c), f(s)) + (1 - alpha) * f(c) )
    Pre-computed features can be passed in to speed up alpha sweeps.
    """
    assert 0.0 <= alpha <= 1.0
    fc = content_feat if content_feat is not None else encode(content)
    fs = style_feat if style_feat is not None else encode(style)
    t = adain(fc, fs)
    t = alpha * t + (1.0 - alpha) * fc
    return decoder(t).clamp(0.0, 1.0)

In [ ]:
def load_image(path: Path, max_size: int = IMAGE_SIZE) -> torch.Tensor:
    """Load any image as RGB, resize so the longest side = max_size, and snap to multiples of 8."""
    img = Image.open(path).convert("RGB")         # handles RGBA PNG / WEBP
    w, h = img.size
    scale = max_size / max(w, h)
    new_w = max(8, int(round(w * scale / 8)) * 8)  # 3 pooling stages -> sizes divisible by 8
    new_h = max(8, int(round(h * scale / 8)) * 8)
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    return TF.to_tensor(img).unsqueeze(0).to(DEVICE)


def to_numpy(t: torch.Tensor) -> np.ndarray:
    return t.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()


def save_tensor(t: torch.Tensor, path: Path):
    Image.fromarray((to_numpy(t) * 255).round().astype(np.uint8)).save(path)

: 

## 6. Single-Alpha Experiment

Runs AdaIN using the `ALPHA`, `SINGLE_CONTENT` and `SINGLE_STYLE` values from the Configuration cell.

In [ ]:
content_img = load_image(CONTENT_DIR / SINGLE_CONTENT)
style_img = load_image(STYLE_DIR / SINGLE_STYLE)

t0 = time.perf_counter()
output = stylize(content_img, style_img, alpha=ALPHA)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
elapsed = (time.perf_counter() - t0) * 1000

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [content_img, style_img, output],
                          [f"Content: {SINGLE_CONTENT}", f"Style: {SINGLE_STYLE}", f"AdaIN output (alpha = {ALPHA})"]):
    ax.imshow(to_numpy(img)); ax.set_title(title); ax.axis("off")
plt.suptitle(f"Single-pass AdaIN stylization | inference time: {elapsed:.1f} ms on {DEVICE}", y=1.02)
plt.tight_layout(); plt.show()

### Interactive Alpha Slider

Move the slider to change $\alpha$ live. Encoder features are cached, so each update only runs the decoder.

> Requires `ipywidgets` (`pip install ipywidgets`). If it is not installed, change `ALPHA` in Section 1 instead.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    _feat_cache = {}

    def _cached_feats(c_name, s_name):
        key = (c_name, s_name)
        if key not in _feat_cache:
            c = load_image(CONTENT_DIR / c_name)
            s = load_image(STYLE_DIR / s_name)
            _feat_cache[key] = (c, encode(c), encode(s), s)
        return _feat_cache[key]

    def _show(alpha, content_name, style_name):
        c, fc, fs, s = _cached_feats(content_name, style_name)
        out = stylize(c, s, alpha=alpha, content_feat=fc, style_feat=fs)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, img, title in zip(axes, [c, s, out], ["Content", "Style", f"Output (alpha = {alpha:.2f})"]):
            ax.imshow(to_numpy(img)); ax.set_title(title); ax.axis("off")
        plt.tight_layout(); plt.show()

    widgets.interact(
        _show,
        alpha=widgets.FloatSlider(value=ALPHA, min=0.0, max=1.0, step=0.05, description="alpha",
                                  continuous_update=False, readout_format=".2f"),
        content_name=widgets.Dropdown(options=sorted(p.name for p in CONTENT_DIR.iterdir()),
                                      value=SINGLE_CONTENT, description="content"),
        style_name=widgets.Dropdown(options=sorted(p.name for p in STYLE_DIR.iterdir()),
                                    value=SINGLE_STYLE, description="style"),
    )
except ImportError:
    print("ipywidgets not installed. Edit ALPHA in the Configuration cell, or run: pip install ipywidgets")

## 7. Evaluation Metrics

To measure the trade-off objectively, we evaluate each output $\hat{x} = T(c, s, \alpha)$ along **two axes**.

### 7.1 Content Preservation (↑ better preservation = ↓ loss / ↑ similarity)

| Metric | Definition | Direction |
| :-- | :-- | :--: |
| **Content loss** $\mathcal{L}_c$ | $\lVert f(\hat{x}) - f(c) \rVert_2^2$ at `relu4_1` (perceptual distance to the original content) | ↓ |
| **SSIM** | Structural Similarity between $\hat{x}$ and $c$ (Wang et al., 2004) | ↑ |
| **Edge similarity** | Pearson correlation of Sobel gradient magnitudes of $\hat{x}$ and $c$ | ↑ |
| **LPIPS** *(optional)* | Learned perceptual distance (Zhang et al., 2018), computed only if `lpips` is installed | ↓ |

### 7.2 Style Fidelity (↑ fidelity = ↓ loss)

| Metric | Definition | Direction |
| :-- | :-- | :--: |
| **AdaIN style loss** $\mathcal{L}_s$ | $\sum_{i=1}^{4} \lVert \mu(\phi_i(\hat{x})) - \mu(\phi_i(s)) \rVert_2^2 + \lVert \sigma(\phi_i(\hat{x})) - \sigma(\phi_i(s)) \rVert_2^2$ over `relu1_1`…`relu4_1` (the loss used to train AdaIN) | ↓ |
| **Gram style loss** $\mathcal{L}_{gram}$ | $\sum_i \lVert G(\phi_i(\hat{x})) - G(\phi_i(s)) \rVert_F^2$, the classic Gatys et al. texture statistic | ↓ |

> **Note on α = 0:** the output is $g(f(c))$, i.e. the decoder's *reconstruction* of the content image. This is not exactly the original, so $\mathcal{L}_c > 0$ and SSIM < 1 even at α = 0. That value represents the **reconstruction floor** of the model.

In [ ]:
def gram_matrix(feat: torch.Tensor) -> torch.Tensor:
    n, c, h, w = feat.shape
    f = feat.reshape(n, c, h * w)
    return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)


def _gaussian_window(size: int = 11, sigma: float = 1.5, channels: int = 3) -> torch.Tensor:
    coords = torch.arange(size, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).unsqueeze(0)
    window = (g.t() @ g).expand(channels, 1, size, size).contiguous()
    return window.to(DEVICE)


_SSIM_WIN = _gaussian_window()


def ssim(x: torch.Tensor, y: torch.Tensor) -> float:
    """Standard SSIM (Gaussian window 11, sigma 1.5) on RGB images in [0, 1]."""
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    pad = _SSIM_WIN.shape[-1] // 2
    mu_x = F.conv2d(x, _SSIM_WIN, padding=pad, groups=3)
    mu_y = F.conv2d(y, _SSIM_WIN, padding=pad, groups=3)
    sxx = F.conv2d(x * x, _SSIM_WIN, padding=pad, groups=3) - mu_x ** 2
    syy = F.conv2d(y * y, _SSIM_WIN, padding=pad, groups=3) - mu_y ** 2
    sxy = F.conv2d(x * y, _SSIM_WIN, padding=pad, groups=3) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sxx + syy + c2))
    return ssim_map.mean().item()


_SOBEL_X = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=DEVICE).view(1, 1, 3, 3)
_SOBEL_Y = _SOBEL_X.transpose(2, 3)


def edge_similarity(x: torch.Tensor, y: torch.Tensor) -> float:
    """Pearson correlation between Sobel gradient-magnitude maps of x and y."""
    def grad_mag(img):
        gray = (0.299 * img[:, 0:1] + 0.587 * img[:, 1:2] + 0.114 * img[:, 2:3])
        gx = F.conv2d(gray, _SOBEL_X, padding=1)
        gy = F.conv2d(gray, _SOBEL_Y, padding=1)
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12).flatten()
    a, b = grad_mag(x), grad_mag(y)
    a, b = a - a.mean(), b - b.mean()
    return (a @ b / (a.norm() * b.norm() + 1e-12)).item()


try:
    import lpips
    _lpips_fn = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
    print("LPIPS enabled.")
except ImportError:
    _lpips_fn = None
    print("LPIPS not installed (optional): pip install lpips")


@torch.no_grad()
def evaluate(output, content, style_feats_all, content_feat_r41):
    """Compute all trade-off metrics for one stylized output."""
    out_feats = encode(output, all_layers=True)

    # ---- content preservation ----
    content_loss = F.mse_loss(out_feats[-1], content_feat_r41).item()
    metrics = {
        "content_loss": content_loss,
        "ssim": ssim(output, content),
        "edge_similarity": edge_similarity(output, content),
        "lpips": (_lpips_fn(output * 2 - 1, content * 2 - 1).item() if _lpips_fn is not None else np.nan),
    }

    # ---- style fidelity ----
    adain_style, gram_style = 0.0, 0.0
    for fo, fs in zip(out_feats, style_feats_all):
        mo, so = calc_mean_std(fo)
        ms, ss = calc_mean_std(fs)
        adain_style += (F.mse_loss(mo, ms) + F.mse_loss(so, ss)).item()
        gram_style += F.mse_loss(gram_matrix(fo), gram_matrix(fs)).item()
    metrics["style_loss"] = adain_style
    metrics["gram_style_loss"] = gram_style
    return metrics


# Quick sanity check on the single-run output
_m = evaluate(output, content_img, encode(style_img, all_layers=True), encode(content_img))
pd.DataFrame([_m], index=[f"alpha={ALPHA}"]).round(4)

## 8. Systematic α-Sweep Experiment

For every content–style pair in `SWEEP_CONTENTS × SWEEP_STYLES` and every $\alpha$ in `ALPHA_SWEEP`, we stylize the image and compute all metrics. Encoder features are computed once per pair, so only the decoder runs for each $\alpha$.

In [ ]:
records = []
outputs = {}   # (content, style, alpha) -> tensor, used for the qualitative grid

pairs = [(c, s) for c in SWEEP_CONTENTS for s in SWEEP_STYLES]
print(f"{len(pairs)} pairs x {len(ALPHA_SWEEP)} alphas = {len(pairs) * len(ALPHA_SWEEP)} stylizations\n")

sweep_start = time.perf_counter()
for c_name, s_name in pairs:
    c_img = load_image(CONTENT_DIR / c_name)
    s_img = load_image(STYLE_DIR / s_name)
    fc = encode(c_img)
    fs_r41 = encode(s_img)
    fs_all = encode(s_img, all_layers=True)

    for alpha in ALPHA_SWEEP:
        t0 = time.perf_counter()
        out = stylize(c_img, s_img, alpha=alpha, content_feat=fc, style_feat=fs_r41)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        infer_ms = (time.perf_counter() - t0) * 1000

        m = evaluate(out, c_img, fs_all, fc)
        m.update({"content": c_name, "style": s_name, "alpha": alpha, "inference_ms": infer_ms})
        records.append(m)
        outputs[(c_name, s_name, alpha)] = out.cpu()

        if SAVE_RESULTS:
            save_tensor(out, IMG_OUT_DIR / f"{Path(c_name).stem}__{Path(s_name).stem}__a{alpha:.2f}.png")

    print(f"  done: {c_name:<14} x {s_name:<12}")

df = pd.DataFrame(records)[["content", "style", "alpha", "content_loss", "ssim", "edge_similarity",
                            "lpips", "style_loss", "gram_style_loss", "inference_ms"]]
print(f"\nSweep finished in {time.perf_counter() - sweep_start:.1f} s")

if SAVE_RESULTS:
    df.to_csv(RESULT_DIR / "alpha_sweep_metrics.csv", index=False)
    print("Saved:", RESULT_DIR / "alpha_sweep_metrics.csv")

df.head(len(ALPHA_SWEEP)).round(4)

## 9. Qualitative Analysis: Visual Progression Across α

Each row is a content–style pair, and each column is an $\alpha$ value. Read left to right to see style strength increase.

In [ ]:
grid_alphas = [a for a in ALPHA_SWEEP if round(a * 10) % 2 == 0] or ALPHA_SWEEP   # 0.0, 0.2, ... for readability
n_rows, n_cols = len(pairs), len(grid_alphas) + 1

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.6 * n_rows), squeeze=False)
for r, (c_name, s_name) in enumerate(pairs):
    axes[r, 0].imshow(to_numpy(load_image(STYLE_DIR / s_name, max_size=256)))
    axes[r, 0].set_ylabel(f"{Path(c_name).stem}\n× {Path(s_name).stem}", fontsize=9)
    axes[r, 0].set_title("style" if r == 0 else "", fontsize=10)
    for k, a in enumerate(grid_alphas, start=1):
        axes[r, k].imshow(to_numpy(outputs[(c_name, s_name, a)]))
        if r == 0:
            axes[r, k].set_title(f"α = {a:.1f}", fontsize=10)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

plt.suptitle("AdaIN outputs under increasing style strength α", fontsize=14, y=1.005)
plt.tight_layout()
if SAVE_RESULTS:
    plt.savefig(FIG_DIR / "qualitative_alpha_grid.png", bbox_inches="tight", dpi=150)
plt.show()

## 10. Quantitative Analysis: Metrics vs. α (RQ1)

Solid lines show the mean across all pairs, and shaded bands show ±1 standard deviation.

In [ ]:
metric_info = [
    ("content_loss", "Content loss (relu4_1) ↓", "tab:blue"),
    ("ssim", "SSIM vs. content ↑", "tab:blue"),
    ("edge_similarity", "Edge similarity vs. content ↑", "tab:blue"),
    ("style_loss", "AdaIN style loss (mean/std) ↓", "tab:red"),
    ("gram_style_loss", "Gram style loss ↓", "tab:red"),
]
if df["lpips"].notna().any():
    metric_info.insert(3, ("lpips", "LPIPS vs. content ↓", "tab:blue"))

agg = df.groupby("alpha")[[m[0] for m in metric_info]].agg(["mean", "std"])
n = len(metric_info)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 3.8))
for ax, (col, label, color) in zip(axes, metric_info):
    mean, std = agg[(col, "mean")], agg[(col, "std")].fillna(0)
    ax.plot(mean.index, mean.values, "o-", color=color, lw=2)
    ax.fill_between(mean.index, mean - std, mean + std, color=color, alpha=0.15)
    ax.set_xlabel("α (style strength)"); ax.set_title(label, fontsize=10)
plt.suptitle("Blue = content preservation | Red = style fidelity", y=1.03)
plt.tight_layout()
if SAVE_RESULTS:
    plt.savefig(FIG_DIR / "metrics_vs_alpha.png", bbox_inches="tight", dpi=150)
plt.show()

summary = df.groupby("alpha")[[m[0] for m in metric_info]].mean()
summary.round(4)

In [ ]:
# Hypothesis check (H1, H2): monotonicity via Spearman rank correlation with alpha, per pair
from scipy.stats import spearmanr

rows = []
for (c_name, s_name), g in df.groupby(["content", "style"]):
    row = {"content": c_name, "style": s_name}
    for col in ["content_loss", "ssim", "style_loss", "gram_style_loss"]:
        row[f"rho_{col}"] = spearmanr(g["alpha"], g[col]).correlation
    rows.append(row)
mono = pd.DataFrame(rows)

print("Spearman rho with alpha (+1 = perfectly increasing, -1 = perfectly decreasing)")
print(f"  H1 style_loss decreases   : mean rho = {mono['rho_style_loss'].mean():+.3f}")
print(f"  H2 content_loss increases : mean rho = {mono['rho_content_loss'].mean():+.3f}")
print(f"  H2 SSIM decreases         : mean rho = {mono['rho_ssim'].mean():+.3f}")
mono.round(3)

## 11. Trade-off Curve (RQ2)

Each curve plots **content loss** against **style loss** for one pair, with $\alpha$ annotated along the curve. A perfectly linear trade-off would be a straight line. Curvature reveals where increasing $\alpha$ costs more content than it gains in style.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap("tab10")
for i, ((c_name, s_name), g) in enumerate(df.groupby(["content", "style"])):
    g = g.sort_values("alpha")
    ax.plot(g["style_loss"], g["content_loss"], "o-", color=cmap(i % 10), ms=4, lw=1.5,
            label=f"{Path(c_name).stem} × {Path(s_name).stem}")
    for _, r in g.iloc[[0, len(g) // 2, -1]].iterrows():
        ax.annotate(f"α={r['alpha']:.1f}", (r["style_loss"], r["content_loss"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points", color=cmap(i % 10))
ax.set_xscale("log")
ax.set_xlabel("AdaIN style loss (log scale) → lower = higher style fidelity")
ax.set_ylabel("Content loss → lower = better content preservation")
ax.set_title("Style fidelity vs. content preservation trade-off")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
if SAVE_RESULTS:
    plt.savefig(FIG_DIR / "tradeoff_curve.png", bbox_inches="tight", dpi=150)
plt.show()

## 12. Finding the Balanced Style Strength α* (RQ3)

Content loss and style loss have very different scales, so for each pair we **min-max normalize** them across α into scores in $[0, 1]$:

$$
S_{content}(\alpha) = 1 - \frac{\mathcal{L}_c(\alpha) - \min \mathcal{L}_c}{\max \mathcal{L}_c - \min \mathcal{L}_c}, \qquad
S_{style}(\alpha) = 1 - \frac{\mathcal{L}_s(\alpha) - \min \mathcal{L}_s}{\max \mathcal{L}_s - \min \mathcal{L}_s}
$$

We define the **balance score** as the harmonic mean, which rewards outputs that do well on *both* criteria:

$$
B(\alpha) = \frac{2\, S_{content}\, S_{style}}{S_{content} + S_{style}}, \qquad \alpha^* = \arg\max_\alpha B(\alpha)
$$

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0


parts = []
for _, g in df.groupby(["content", "style"]):
    g = g.sort_values("alpha").copy()
    g["S_content"] = 1 - minmax(g["content_loss"])
    g["S_style"] = 1 - minmax(g["style_loss"])
    g["balance"] = 2 * g["S_content"] * g["S_style"] / (g["S_content"] + g["S_style"] + 1e-12)
    parts.append(g)
df_norm = pd.concat(parts)

best = df_norm.loc[df_norm.groupby(["content", "style"])["balance"].idxmax(),
                   ["content", "style", "alpha", "S_content", "S_style", "balance", "ssim"]]
best = best.rename(columns={"alpha": "alpha_star"}).reset_index(drop=True)

mean_curve = df_norm.groupby("alpha")[["S_content", "S_style", "balance"]].mean()
global_alpha_star = mean_curve["balance"].idxmax()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
ax.plot(mean_curve.index, mean_curve["S_content"], "o-", color="tab:blue", lw=2, label="Content preservation score")
ax.plot(mean_curve.index, mean_curve["S_style"], "o-", color="tab:red", lw=2, label="Style fidelity score")
ax.plot(mean_curve.index, mean_curve["balance"], "s--", color="tab:green", lw=2, label="Balance (harmonic mean)")
ax.axvline(global_alpha_star, color="gray", ls=":", lw=1.5)
ax.text(global_alpha_star, 0.02, f"  α* = {global_alpha_star:.1f}", color="gray")
ax.set_xlabel("α"); ax.set_ylabel("normalized score"); ax.set_title("Mean normalized trade-off across all pairs")
ax.legend(fontsize=8)

ax = axes[1]
pivot = best.pivot(index="content", columns="style", values="alpha_star")
im = ax.imshow(pivot.values, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=20)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.1f}", ha="center", va="center", color="white", fontweight="bold")
ax.grid(False); ax.set_title("Balanced α* per content–style pair")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
if SAVE_RESULTS:
    plt.savefig(FIG_DIR / "balanced_alpha.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"Global balanced alpha* = {global_alpha_star:.2f}")
best.round(3)

## 13. Style Sensitivity Analysis

Some styles degrade content much faster than others. For each style, we measure:

- **Content sensitivity** = mean slope $\Delta \mathcal{L}_c / \Delta\alpha$ (higher = content breaks down faster).
- **Style gain** = relative style-loss reduction from α = 0 to α = 1.
- **Mean α\*** = average balanced strength for that style.

In [ ]:
sens_rows = []
for s_name, g in df.groupby("style"):
    slopes, gains = [], []
    for _, gp in g.groupby("content"):
        gp = gp.sort_values("alpha")
        slopes.append(np.polyfit(gp["alpha"], gp["content_loss"], 1)[0])
        s0, s1 = gp["style_loss"].iloc[0], gp["style_loss"].iloc[-1]
        gains.append((s0 - s1) / s0 * 100 if s0 > 0 else np.nan)
    sens_rows.append({
        "style": s_name,
        "content_sensitivity (dLc/dα)": np.mean(slopes),
        "style_gain_%": np.mean(gains),
        "ssim_at_α=1": g[g["alpha"] == max(ALPHA_SWEEP)]["ssim"].mean(),
        "mean_alpha_star": best[best["style"] == s_name]["alpha_star"].mean(),
    })
sensitivity = pd.DataFrame(sens_rows).sort_values("content_sensitivity (dLc/dα)", ascending=False)
if SAVE_RESULTS:
    sensitivity.to_csv(RESULT_DIR / "style_sensitivity.csv", index=False)
    best.to_csv(RESULT_DIR / "balanced_alpha_per_pair.csv", index=False)
sensitivity.round(3)

## 14. Summary of Findings

In [ ]:
a_min, a_max = min(ALPHA_SWEEP), max(ALPHA_SWEEP)
s_lo, s_hi = summary.loc[a_min], summary.loc[a_max]

print("=" * 72)
print(" SUMMARY: AdaIN style strength trade-off")
print("=" * 72)
print(f" Pairs evaluated : {len(pairs)}   |   alpha values: {len(ALPHA_SWEEP)}   |   device: {DEVICE}")
print(f" Mean inference  : {df['inference_ms'].mean():.1f} ms per image (decoder only)\n")
print(f" From alpha={a_min:.1f} to alpha={a_max:.1f} (mean over pairs):")
print(f"   Content loss    : {s_lo['content_loss']:.4f} -> {s_hi['content_loss']:.4f}")
print(f"   SSIM            : {s_lo['ssim']:.4f} -> {s_hi['ssim']:.4f}")
print(f"   Edge similarity : {s_lo['edge_similarity']:.4f} -> {s_hi['edge_similarity']:.4f}")
print(f"   AdaIN style loss: {s_lo['style_loss']:.4f} -> {s_hi['style_loss']:.4f}")
print(f"   Gram style loss : {s_lo['gram_style_loss']:.6f} -> {s_hi['gram_style_loss']:.6f}\n")
print(f" H1 (style loss decreases)   mean Spearman rho = {mono['rho_style_loss'].mean():+.3f}")
print(f" H2 (content loss increases) mean Spearman rho = {mono['rho_content_loss'].mean():+.3f}")
print(f" H3 balanced alpha*: global = {global_alpha_star:.2f}, "
      f"per-pair range = [{best['alpha_star'].min():.1f}, {best['alpha_star'].max():.1f}], "
      f"std = {best['alpha_star'].std():.3f}")
most = sensitivity.iloc[0]["style"]; least = sensitivity.iloc[-1]["style"]
print(f" Most content-destructive style : {most}")
print(f" Least content-destructive style: {least}")
if SAVE_RESULTS:
    print(f"\n All artifacts saved to: {RESULT_DIR}")
print("=" * 72)

### Discussion Template

Use the printed summary above to complete the discussion for the paper:

1. **RQ1: Behaviour of the metrics.** Describe how content loss / SSIM / edge similarity and style losses evolve with α. Report the Spearman correlations as evidence for H1 and H2.
2. **RQ2: Shape of the trade-off.** Use the trade-off curve (Section 11) to discuss whether the curve is convex, and identify any α range where style gains saturate while content loss keeps increasing.
3. **RQ3: Style dependence.** Use the α\* heatmap and the sensitivity table to discuss why some styles (e.g., high-frequency, strongly textured artworks) need a lower α to preserve content than smoother styles do.
4. **Practical recommendation.** Suggest a default α (global α\*), and recommend style-adaptive α selection when content fidelity matters.
5. **Limitations.** Small image set, a single pretrained decoder, the reconstruction floor at α = 0, the arbitrary choice of normalization for the balance score, and no human perceptual study.

### Possible Extensions

- Add a **user study** or **CLIP-based aesthetic score** to complement the automatic metrics.
- Compare AdaIN against **WCT**, **SANet** or **AdaAttN** at matched style strengths.
- Explore **multi-style interpolation** with weights $w_k$ (AdaIN supports convex combinations of style features).
- Apply **spatial control** (masks) to use different α values for foreground and background.

## References

1. Huang, X., & Belongie, S. (2017). *Arbitrary Style Transfer in Real-time with Adaptive Instance Normalization*. ICCV.
2. Gatys, L. A., Ecker, A. S., & Bethge, M. (2016). *Image Style Transfer Using Convolutional Neural Networks*. CVPR.
3. Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). *Image Quality Assessment: From Error Visibility to Structural Similarity*. IEEE TIP.
4. Zhang, R., Isola, P., Efros, A. A., Shechtman, E., & Wang, O. (2018). *The Unreasonable Effectiveness of Deep Features as a Perceptual Metric*. CVPR.
5. Simonyan, K., & Zisserman, A. (2015). *Very Deep Convolutional Networks for Large-Scale Image Recognition*. ICLR.